
# SIAM — Replication Notebook (LOSO Validation)

This notebook reproduces a **Leave-One-Subject-Out (LOSO)** validation using the **synthetic** example dataset included in the repository.
It trains a Random Forest classifier, aggregates out-of-fold predictions, computes key metrics (Accuracy, ROC AUC, PR AUC, Brier score), 
performs simple bootstrap confidence intervals, and **exports** figures/metrics to the `results/` directory:
- `results/roc_curve.png`
- `results/pr_curve.png`
- `results/calibration_curve.png`
- `results/metrics_bootstrap.csv`
- `results/predictions_oof.csv`

> **Note**: The dataset is synthetic and intended **only** for demonstration.


In [ ]:

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, roc_curve, 
                             precision_recall_curve, average_precision_score,
                             brier_score_loss, confusion_matrix, classification_report)
from sklearn.calibration import calibration_curve

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths (adjust if running outside the repo)
DATA_PATH = 'data/example_dataset.csv'   # relative to repo root
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
df.head()


In [ ]:

# Parse subject ID from 'archivo' like 'Subject01_NORMAL.csv' -> 'Subject01'
def parse_subject(s):
    m = re.match(r'(Subject\d+)_', str(s))
    return m.group(1) if m else str(s).split('.')[0]

df['subject_id'] = df['archivo'].apply(parse_subject)

# Features and label
X = df.drop(columns=['clase','archivo','subject_id'])
y = df['clase'].map({'ALZ':1, 'CONTROL':0})  # binary encoding
subjects = df['subject_id'].values

print('Unique subjects:', df['subject_id'].nunique())
df[['subject_id','clase']].head()


In [ ]:

unique_subjects = df['subject_id'].unique()

y_true_all = []
y_prob_all = []
y_pred_all = []
subj_all = []

for subj in unique_subjects:
    test_idx = (subjects == subj)
    train_idx = ~test_idx
    
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    clf = RandomForestClassifier(
        n_estimators=300, 
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)
    
    prob = clf.predict_proba(X_test)[:,1]
    pred = (prob >= 0.5).astype(int)
    
    y_true_all.extend(y_test.tolist())
    y_prob_all.extend(prob.tolist())
    y_pred_all.extend(pred.tolist())
    subj_all.extend(df.loc[test_idx, 'subject_id'].tolist())

y_true_all = np.array(y_true_all)
y_prob_all = np.array(y_prob_all)
y_pred_all = np.array(y_pred_all)

acc = accuracy_score(y_true_all, y_pred_all)
roc_auc = roc_auc_score(y_true_all, y_prob_all)
pr_auc = average_precision_score(y_true_all, y_prob_all)
brier = brier_score_loss(y_true_all, y_prob_all)

print(f'LOSO Accuracy: {acc:.3f}')
print(f'LOSO ROC AUC: {roc_auc:.3f}')
print(f'LOSO PR AUC : {pr_auc:.3f}')
print(f'Brier score : {brier:.3f}')

# Save OOF predictions
oof_df = pd.DataFrame({
    'subject_id': subj_all,
    'y_true': y_true_all,
    'y_pred': y_pred_all,
    'y_prob': y_prob_all
})
oof_df.to_csv(os.path.join(RESULTS_DIR, 'predictions_oof.csv'), index=False)
oof_df.head()


In [ ]:

def bootstrap_ci(y_true, y_prob, y_pred, n_boot=500, alpha=0.05, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    acc_samples, auc_samples, pr_samples, brier_samples = [], [], [], []
    
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        yp = y_prob[idx]
        yd = y_pred[idx]
        try:
            acc_samples.append(accuracy_score(yt, yd))
            auc_samples.append(roc_auc_score(yt, yp))
            pr_samples.append(average_precision_score(yt, yp))
            brier_samples.append(brier_score_loss(yt, yp))
        except ValueError:
            # In case a sample has a single class, skip ROC/PR computation
            continue
    
    def ci(arr):
        arr = np.array(arr)
        lower = np.quantile(arr, alpha/2)
        upper = np.quantile(arr, 1-alpha/2)
        return float(np.mean(arr)), float(lower), float(upper)
    
    return {
        'accuracy': ci(acc_samples),
        'roc_auc': ci(auc_samples),
        'pr_auc': ci(pr_samples),
        'brier': ci(brier_samples)
    }

stats = bootstrap_ci(y_true_all, y_prob_all, y_pred_all, n_boot=1000, alpha=0.05)
stats


In [ ]:

# Export metrics to CSV
rows = []
for k, (mean, lo, hi) in stats.items():
    rows.append({'metric': k, 'mean': mean, 'ci95_lower': lo, 'ci95_upper': hi})
metrics_df = pd.DataFrame(rows)
metrics_path = os.path.join(RESULTS_DIR, 'metrics_bootstrap.csv')
metrics_df.to_csv(metrics_path, index=False)
metrics_df


In [ ]:

fpr, tpr, _ = roc_curve(y_true_all, y_prob_all)
plt.figure()
plt.plot(fpr, tpr, label=f'LOSO ROC (AUC={roc_auc:.2f})')
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — LOSO')
plt.legend()
plt.grid(True)
fig_path = os.path.join(RESULTS_DIR, 'roc_curve.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
fig_path


In [ ]:

prec, rec, _ = precision_recall_curve(y_true_all, y_prob_all)
plt.figure()
plt.plot(rec, prec, label=f'LOSO PR (AP={pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — LOSO')
plt.legend()
plt.grid(True)
fig_path = os.path.join(RESULTS_DIR, 'pr_curve.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
fig_path


In [ ]:

prob_true, prob_pred = calibration_curve(y_true_all, y_prob_all, n_bins=5, strategy='quantile')
plt.figure()
plt.plot(prob_pred, prob_true, marker='o', label='LOSO')
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration Curve — LOSO')
plt.legend()
plt.grid(True)
fig_path = os.path.join(RESULTS_DIR, 'calibration_curve.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
fig_path


In [ ]:

cm = confusion_matrix(y_true_all, y_pred_all)
print('Confusion Matrix:\n', cm)
print('\nClassification Report:\n', classification_report(y_true_all, y_pred_all, target_names=['CONTROL','ALZ']))
